# House Price Prediction with Ridge Regression

This notebook uses the UCI Real Estate Valuation dataset to predict house price per unit area. It loads the data, splits it into training and test sets, trains a Ridge Regression pipeline, evaluates its performance, and predicts the price for an example property.

**Note:** The target is price per unit area, not the total price of a house.

In [ ]:
%pip install ucimlrepo scikit-learn pandas


In [ ]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 4775

# Load the UCI Real Estate Valuation dataset (dataset ID 477)
real_estate = fetch_ucirepo(id=477)
X = real_estate.data.features.copy()
y = real_estate.data.targets.iloc[:, 0].copy()

X.columns = [
    'transaction_date',
    'house_age',
    'distance_mrt',
    'num_convenience_stores',
    'latitude',
    'longitude',
]
y.name = 'house_price_unit_area'

print('Dataset shape:', X.shape)
display(pd.concat([X, y], axis=1).head())

In [ ]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED
)

# Imputation and scaling happen inside the pipeline to avoid data leakage.
model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('regressor', Ridge(alpha=1.0)),
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)
print('Model trained. Test rows:', len(X_test))

In [ ]:
# Evaluate predictions on the held-out test set
mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions) ** 0.5
r2 = r2_score(y_test, predictions)

print(f'MAE:  {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R²:   {r2:.3f}')

In [ ]:
# Predict price per unit area for an example property
example_house = pd.DataFrame([{
    'transaction_date': 2013.25,
    'house_age': 20.0,
    'distance_mrt': 500.0,
    'num_convenience_stores': 5,
    'latitude': 24.98,
    'longitude': 121.54,
}])

predicted_price = model.predict(example_house)[0]
print(f'Predicted price per unit area: {predicted_price:.2f}')
print('This is a dataset-based estimate, not a guaranteed market price.')